```text
 ___ _____ _____ ____   ___
|_ _|_   _| ____/ ___| / _ \
 | |  | | |  _| \___ \| | | |
 | |  | | | |___ ___) | |_| |
|___| |_| |_____|____/ \___/
```

# 🎓 Examen final — Del mensaje al modelo

**Nimbo Labs · Proyecto Nimbo SMS · Clasificación de mensajes**

## 🧩 Planteamiento del problema

**Nimbo Labs**, una empresa ficticia que transforma prototipos de ciencia de datos en aplicaciones utilizables, está desarrollando una solución de mensajería. Su equipo desarrolla **Nimbo SMS**, un prototipo para ayudar a identificar mensajes potencialmente no solicitados. Una persona debe poder pegar un SMS escrito en inglés en una interfaz y obtener una clasificación mediante una API.

La salida debe distinguir `ham` (mensaje legítimo) y `spam` (mensaje no solicitado), e identificar la versión del modelo que produjo la predicción. La aplicación apoya la decisión de la persona; no bloquea ni elimina mensajes automáticamente.


## 📚 Datos disponibles

Dispones de [SMS Spam Collection de UCI](https://archive.ics.uci.edu/dataset/228/sms%2Bspam%2Bcollection), un corpus histórico de mensajes en inglés etiquetados como `ham` o `spam`. La entrada del modelo es el contenido del mensaje; la etiqueta sirve como objetivo durante entrenamiento y evaluación.

**Entrada del sistema:** texto de un SMS en inglés.

**Salida del sistema:** clasificación estimada (`ham` o `spam`) y versión del modelo utilizado.


## 💡 Contexto de uso

Los errores tienen consecuencias distintas: un falso positivo marca como spam un mensaje legítimo; un falso negativo deja pasar un mensaje no solicitado. La utilidad del prototipo depende de interpretar esos errores y de mantener consistencia entre el modelo evaluado y el utilizado por la aplicación.

El corpus histórico permite desarrollar y evaluar el prototipo, pero sus resultados no demuestran por sí solos eficacia con mensajes actuales, otros idiomas o todos los tipos de fraude.


# $\color{#dda}{\text{1. Problema y diseño}}$

**Objetivo general.** Construir un prototipo reproducible que, a partir del texto de un SMS en inglés, estime si es `ham` o `spam` e informe qué versión del modelo produjo la estimación, conservando el mismo procesamiento entre evaluación y servicio.

**Objetivos específicos.**

1. Adquirir y depurar el corpus de UCI de forma reproducible y separar entrenamiento, validación y prueba sin mensajes equivalentes compartidos.
2. Comparar un baseline con dos familias de modelos (regresión logística y Multinomial Naive Bayes) usando una métrica definida antes del tuning.
3. Registrar experimentos y dos versiones del modelo en Databricks (MLflow + Unity Catalog) con aliases `champion` / `challenger`.
4. Coordinar todo el entrenamiento desde terminal con Prefect, sin Jupyter.
5. Servir el champion con una API FastAPI que cumpla el contrato HTTP y consultarla desde una interfaz Streamlit, ambas en contenedores separados.

**Usuario.** Una persona que recibió un SMS dudoso y quiere una segunda opinión. La salida **apoya** su decisión: no bloquea ni borra mensajes.

**Entrada / salida.** Texto libre de 1 a 2,000 caracteres → `label` (`ham`/`spam`), `model_name`, `model_version`.

**Alcance.** SMS en inglés parecidos a los del corpus (2011). Fuera de alcance: otros idiomas, fraude actual, bloqueo automático, monitoreo y reentrenamiento en producción.

**Arquitectura.**

```text
UCI ZIP ──▶ src/datos.py ──▶ data/processed/{train,validation,test}.parquet
                                   │
             src/train_flow.py (Prefect) ── Optuna ── MLflow (Databricks)
                                   │                       │
                         reports/metricas.csv     Unity Catalog: v1 @champion, v2 @challenger
                                                           │  (se resuelve al iniciar)
   Persona ──▶ UI Streamlit (ui/, :8560) ──HTTP──▶ API FastAPI (api/, :8060)
```

| Componente | Herramienta | Por qué |
|---|---|---|
| Ambiente | uv + Python 3.12 | Lock único con grupos por servicio ([D1](reports/decisiones-e-ia.md#d1-ambiente-un-solo-uvlock-con-grupos-por-componente)) |
| Representación y modelos | scikit-learn | `Pipeline` serializable que incluye TF-IDF y clasificador |
| Tuning | Optuna (TPE) | Búsqueda guiada con semilla; ya usada en la tarea 5 |
| Tracking y registro | MLflow + Databricks | Camino de referencia; aliases en Unity Catalog |
| Coordinación | Prefect 3 | Tasks con reintentos y estado visible |
| Servicio | FastAPI + Pydantic / Streamlit | Validación declarativa del contrato / interfaz mínima |
| Contenedores | Docker Compose | Un servicio por imagen, red interna |


Configuración del informe: sólo importa los módulos de `src/` y lee resultados guardados; **no repite tuning ni registro**.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))
import datos
import informe

pd = informe.pd
pd.set_option("display.max_colwidth", 80)
particiones = datos.cargar_particiones()
resumen = informe.resumen_datos()
train = particiones["train"]

# $\color{#dda}{\text{2. Datos, particiones y EDA}}$

**Fuente.** SMS Spam Collection, Almeida e Hidalgo (2011), UCI, licencia CC BY 4.0, DOI 10.24432/C5CC84. La descarga y depuración están en [src/datos.py](src/datos.py); el resumen sin mensajes, en [reports/datos-resumen.json](reports/datos-resumen.json). El corpus no se publica: `data/` está en `.gitignore`.


In [2]:
pd.Series({**resumen["descarga"], "licencia": resumen["fuente"]["licencia"], "doi": resumen["fuente"]["doi"]}).to_frame("valor")

,valor
url,https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip
fecha_utc,2026-11-10T19:58:51+00:00
sha256_corpus,7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d
licencia,CC BY 4.0
doi,10.24432/C5CC84


**Calidad.** Se validan etiquetas (sólo `ham`/`spam`), vacíos, contradicciones y duplicados con la clave `texto.strip().lower()`. Una clave contradictoria (mismo texto con ambas etiquetas) se excluiría completa porque no hay forma de saber cuál etiqueta es correcta; un duplicado conserva su primera aparición.

In [3]:
informe.tabla_depuracion(resumen)

,Paso,Conteo
0,Registros leídos,5574
1,Textos vacíos eliminados,0
2,Claves contradictorias (ham y spam),0
3,Duplicados eliminados,415
4,de ellos ham,310
5,de ellos spam,105
6,Registros limpios,5159


Se leyeron **5,574** registros, igual que el archivo revisado del enunciado. No hubo vacíos ni contradicciones; los **415 duplicados** (8 % del corpus) son mucho más frecuentes en spam (105 de 747, 14 %) que en ham (310 de 4,827, 6 %): las campañas de spam repiten el mismo texto. Sin deduplicar, esas copias caerían en entrenamiento y prueba a la vez e inflarían la evaluación.

**Particiones.** 60/20/20 estratificado por etiqueta con semilla 42, **después** de deduplicar. La estratificación mantiene la proporción de spam en las tres; 20 % de prueba deja 128 spam, suficiente para estimar recall con cierta estabilidad.

In [4]:
informe.tabla_particiones(particiones)

,particion,mensajes,ham,spam,proporcion_spam
0,train,3095,2710,385,0.1244
1,validation,1032,903,129,0.1250
2,test,1032,904,128,0.1240


In [5]:
informe.claves_compartidas(particiones)

,par,mensajes_equivalentes_compartidos
0,train ∩ validation,0
1,train ∩ test,0
2,validation ∩ test,0


Cero mensajes equivalentes compartidos en los tres pares: la verificación se recalcula aquí desde los archivos guardados y el flujo la repite en cada ejecución (`verificar_sin_fuga` detiene el flujo si no es cero).

**EDA.** Todo lo que sigue usa **sólo entrenamiento** (3,095 mensajes).

In [6]:
informe.fig_clases(train)

In [7]:
informe.fig_longitud(train)

In [8]:
informe.tabla_rasgos_por_clase(train)

,mediana_caracteres,mediana_palabras,mediana_digitos,con_numero_largo,con_moneda_o_premio,con_url,mayusculas_media
etiqueta,,,,,,,
ham,53.0,11.0,0.0,0.001,0.007,0.003,0.056
spam,148.0,25.0,16.0,0.753,0.387,0.169,0.109


**Observación 1: el spam es largo y está lleno de números.** La mediana de spam es 148 caracteres contra 53 de ham, y 75 % del spam contiene un número de 5 o más dígitos (teléfonos cortos, códigos) contra 0.1 % del ham. El spam busca acercarse al límite de 160 caracteres de un SMS para meter oferta, número y condiciones. *Decisión:* la representación conserva números como tokens (el patrón por defecto de `TfidfVectorizer` los incluye) y no se normalizan ni se eliminan.

**Observación 2: el ham es corto y diverso.** La mitad del ham tiene 11 palabras o menos y usa abreviaturas informales. Un mensaje legítimo muy corto aporta pocas palabras para decidir, así que es de esperar que los errores se concentren en mensajes cortos o atípicos. *Decisión:* se explora `ngram_max=2` (bigramas como "call now") y `min_df` 1–3 para no depender de palabras raras.

**Desbalance.** 12.4 % de spam. Un modelo que siempre responde `ham` obtiene accuracy de 0.875 sin detectar nada, por eso **accuracy es engañosa** aquí y se usa F1 de `spam`. El desbalance se atiende sin remuestreo: se exploran `class_weight="balanced"` (regresión logística) y `fit_prior=False` (Naive Bayes) como hiperparámetros, y el tuning decide si ayudan.


# $\color{#dda}{\text{3. Representación}}$

**Elección:** TF-IDF de palabras (con bigramas opcionales). Para SMS cortos, con vocabulario muy marcado (*free, prize, call, txt*), una bolsa de palabras ponderada es suficiente, rápida y explicable: cada coeficiente se asocia a un término. Los embeddings preentrenados eran opcionales y no se usaron: añaden un modelo de cientos de MB a la imagen de la API sin necesidad demostrada.

**Configuración.** `lowercase=True`, `strip_accents="unicode"`, y como hiperparámetros `ngram_range=(1, 1|2)`, `min_df ∈ {1,2,3}`, `sublinear_tf ∈ {False, True}` ([src/entrenamiento.py](src/entrenamiento.py)).

**Prevención de fuga.** El vectorizador vive **dentro** del `Pipeline` (en un `ColumnTransformer` sobre la columna `text`). En cada trial se crea un pipeline nuevo y `fit` aprende el vocabulario y los IDF **sólo con entrenamiento**; validación y prueba únicamente pasan por `transform`. La depuración previa no aprende nada: sólo compara textos normalizados.

**Consistencia evaluación / inferencia.** El objeto registrado en MLflow es el pipeline completo. La API le pasa el mismo DataFrame `{"text": [...]}` que se usó al evaluar y no aplica ningún procesamiento propio ([D4](reports/decisiones-e-ia.md#d4-representación-dentro-del-modelo-registrado)). La prueba de paridad de la sección 7 lo confirma.


# $\color{#dda}{\text{4. Modelos y selección}}$

**Métrica definida antes del tuning:** F1 de `spam` (principal) y precision de `spam` (desempate) ([D5](reports/decisiones-e-ia.md#d5-métrica-de-selección-definida-antes-del-tuning)). F1 castiga tanto el spam que se escapa (FN) como el ham marcado (FP); ante empate se prefiere menos FP, porque marcar un mensaje legítimo como spam puede hacer que la persona ignore algo importante.

| Modelo | Papel | Espacio de búsqueda (Optuna TPE, semilla 42, 20 trials) |
|---|---|---|
| `DummyClassifier(most_frequent)` | Baseline | Sin búsqueda |
| Regresión logística | Familia 1 (discriminativa) | `C` 0.1–100 log; `class_weight` None/balanced; TF-IDF |
| Multinomial Naive Bayes | Familia 2 (generativa) | `alpha` 0.001–1 log; `fit_prior` T/F; TF-IDF |

**Presupuesto.** Cada ajuste tarda menos de un segundo, así que 20 trials por familia (40 en total, 5 veces el mínimo de 4) cuestan unos minutos, sobre todo por el registro remoto de cada trial. Los rangos en escala logarítmica cubren de regularización/suavizado fuerte a casi nulo.

**Procedimiento.** (1) Tuning de cada familia ajustando con entrenamiento y midiendo en validación. (2) Mejor trial por familia = finalista. (3) Champion = mejor finalista en validación. (4) **Cerrada la selección**, cada finalista y el baseline se reajustan con entrenamiento + validación y se evalúan **una sola vez** en prueba. No se regresa al tuning.


In [9]:
m = informe.metricas()
informe.tabla_comparacion(m)

,conjunto,familia,alias,accuracy,precision_spam,recall_spam,f1_spam,version_modelo
0,validation,multinomial_nb,champion,0.9864,0.9832,0.9070,0.9435,
1,validation,logistic_regression,challenger,0.9855,0.9750,0.9070,0.9398,
2,validation,baseline,,0.8750,0.0000,0.0000,0.0000,
3,test,logistic_regression,challenger,0.9913,0.9917,0.9375,0.9639,2.0
4,test,multinomial_nb,champion,0.9874,0.9832,0.9141,0.9474,1.0
5,test,baseline,,0.8760,0.0000,0.0000,0.0000,


In [10]:
informe.fig_comparacion(m)

In [11]:
pd.DataFrame(informe.seleccion())[["alias", "familia", "trial", "params"]]

,alias,familia,trial,params
0,champion,multinomial_nb,17,"{'ngram_max': 2, 'min_df': 2, 'sublinear_tf': False, 'alpha': 0.061442602442..."
1,challenger,logistic_regression,12,"{'ngram_max': 2, 'min_df': 2, 'sublinear_tf': False, 'C': 96.83345325290557,..."


**Champion: Multinomial Naive Bayes (v1).** En validación obtuvo F1 0.9435 contra 0.9398 de la regresión logística, con precision 0.983 contra 0.975. Ambos finalistas eligieron bigramas y `min_df=2`, y ninguno usó el ajuste de desbalance (`class_weight=None`, `fit_prior=True`): con estos datos, reponderar subía el recall a costa de demasiados FP.

**Prueba.** La regresión logística quedó por encima en prueba (F1 0.9639 contra 0.9474). **El champion no se cambia** ([D6](reports/decisiones-e-ia.md#d6-champion-elegido-en-validación-aunque-pierda-en-prueba)): la prueba es la estimación independiente del desempeño, no un segundo conjunto de selección. Además, 0.004 de F1 en validación equivale a uno o dos mensajes de 129 spam, así que ambos modelos son prácticamente equivalentes. El baseline confirma la advertencia del EDA: accuracy 0.876 con F1 0.

**Límite reconocido** ([D7](reports/decisiones-e-ia.md#d7-c-de-la-regresión-logística-en-el-borde-del-rango-se-documenta-no-se-retunea)): el mejor `C` de la regresión logística (96.8) quedó cerca del borde del rango; no se amplió después de ver prueba.


In [12]:
champion = informe.seleccion()[0]["familia"]
pred = informe.predicciones_test()
matriz = informe.matriz_confusion(pred, champion)
matriz

predicha,ham,spam
real,,
ham,902,2
spam,11,117


In [13]:
informe.fig_matriz(matriz, champion)

In [14]:
informe.conteo_errores(pred, champion)

{'FP': 2, 'FN': 11}

En prueba (904 ham, 128 spam) el champion produce **2 falsos positivos** y **11 falsos negativos**: precision 117/119 = 0.983 y recall 117/128 = 0.914. El modelo es conservador: casi nunca acusa a un mensaje legítimo, pero deja pasar 1 de cada 12 spam. Para una herramienta que **sólo sugiere**, ese balance es el deseado según la métrica de desempate.

**Análisis de errores.** La tabla muestra hasta 3 FP y 3 FN con rasgos numéricos; los textos no se publican (sección 5).

In [15]:
informe.errores(pred, champion)

,id_prueba,tipo,caracteres,palabras,digitos,tiene_numero_largo,tiene_moneda,tiene_url,proporcion_mayusculas
0,286,FP (ham → spam),33,4,0,False,False,False,0.060606
1,771,FP (ham → spam),97,18,0,False,False,False,0.082474
2,33,FN (spam → ham),146,27,20,True,True,False,0.184932
3,41,FN (spam → ham),24,5,7,True,True,False,0.500000
4,263,FN (spam → ham),152,33,2,False,False,False,0.078947


<!-- Remi: corre `uv run python src/ver_errores.py`, lee los mensajes y parafrasea cada uno sin copiarlo ni incluir datos personales. -->

| id | Tipo | Paráfrasis (sin datos sensibles) | Por qué probablemente falló |
|---|---|---|---|
| 286 | FP | Come and claim your prize at 5:00 | Mensaje corto: poca evidencia, basta un término asociado a spam |
| 771 | FP | Mix Lambda, your reward is ready, can you retrieve it tomorrow? | Las palabras prize o reward mandan fácilmente spam |
| 33 | FN | Come and claim your prize at www.scam.com or call 1234567890, all you gotta do is register with your phone number and give us all your money. Please at least answer this message to let us know you're a moron. | Tiene términos de premio |
| 41 | FN | Please lend me 100 USD, I am the old king of Kiribati and need to rebuild my country. | Pocas palabras del vocabulario aprendido |
| 263 | FN | Hello, how are you grandpa? I'm your grandson and crashed my car, please don't let my parents know! I just need a couple grands to fix the old mustang, please please please. | Sin números ni términos de premio: spam con estilo de mensaje personal |

Patrón: los FN que no traen los rasgos típicos (número largo, premio, URL) se parecen a ham para un modelo de bolsa de palabras; los FP son mensajes legítimos cortos donde una sola palabra pesa mucho.


# $\color{#dda}{\text{5. Tracking y registro}}$

| Elemento | Ubicación |
|---|---|
| Experimento | `/Shared/pcd-2026-examen-sms-remih06` (Databricks) |
| Modelo | `workspace.default.pcd_2026_examen_sms_remih06` (Unity Catalog) |
| Aliases | `champion` → v1 (Naive Bayes), `challenger` → v2 (regresión logística) |

**Qué se registra en cada run:** fuente (URL), fecha de descarga y SHA-256 del corpus, semilla, tamaños de partición, métrica de selección y desempate, parámetros del trial, métricas de validación (`val_*`), duración (`duration_seconds`) y tags de rol y familia. Los runs `tuning-<familia>` agrupan sus 20 trials anidados; los runs `final-<familia>` guardan métricas de validación y prueba, la matriz de confusión (`confusion_matrix_test.json`) y el modelo.

**Versiones completas.** Cada versión registrada contiene el pipeline (TF-IDF + clasificador), firma de entrada (`text: string`) y salida (`label`), y un **ejemplo inventado** (`"Can we meet after class?"`); nunca un mensaje del corpus. Cada versión lleva tags con su familia, rol, F1 de validación y prueba y el `run_id` del trial que la originó. La consulta siguiente lee el registro en modo sólo lectura:


In [16]:
informe.versiones_registradas("workspace.default.pcd_2026_examen_sms_remih06")

C:\Users\hecto\OneDrive\Escritorio\ITESO\6to Semestre\PCD\examen-final-pcd-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,version,aliases,familia,val_f1_spam,test_f1_spam,run_id,estado
0,1,champion,multinomial_nb,0.9435,0.9474,895a60be7d434f29ad1fafd8a3045cf4,READY
1,2,challenger,logistic_regression,0.9398,0.9639,4989193707b54c76b3c7461261d952e0,READY


Evidencia persistente: el experimento y el registro viven en Databricks; las capturas están en `reports/evidencias/` (índice en la sección 11) y `reports/metricas.csv` guarda el `run_id` y la versión de cada fila.

# $\color{#dda}{\text{6. Entrenamiento coordinado}}$

```bash
uv run prefect server start --host 127.0.0.1 --port 4200   # opcional: interfaz para ver el flujo
uv run python src/train_flow.py
```

| Task de Prefect | Responsabilidad | Ante fallas |
|---|---|---|
| `adquirir` | Descarga el ZIP oficial sólo si no hay copia local | 3 reintentos (5, 15, 30 s); escribe a `.part` y renombra, así una descarga cortada no deja un ZIP corrupto |
| `preparar_particiones` | Valida, depura, parte, verifica fuga y guarda | `ContratoInvalido` (etiquetas raras, línea sin tabulador, claves compartidas) detiene el flujo sin reintentar |
| `evaluar_baseline` | Baseline en validación | — |
| `tuning-<familia>` | Optuna + runs anidados en MLflow | Si falla una familia, el flujo se detiene antes de seleccionar |
| `seleccionar` | Champion/challenger con validación; guarda `artifacts/seleccion.json` | — |
| `prueba-<familia>` | Reajuste con train+val, una evaluación en prueba, registro del modelo | — |
| `asignar_aliases` | Aliases y tags en Unity Catalog | 2 reintentos (errores de red con Databricks) |
| `guardar_resultados` | `reports/metricas.csv` y predicciones de prueba locales | — |

**Datos existentes.** La copia local en `data/raw/` se reutiliza; volver a correr el flujo no descarga de nuevo y produce las mismas particiones (semilla fija). Antes de empezar, el flujo verifica que existan `DATABRICKS_HOST` y `DATABRICKS_TOKEN`.

**Ejecución.** La corrida completa terminó en `Completed()` en unos 3 minutos y registró las versiones 1 y 2 (evidencias `06-*`). Problema encontrado: con FastAPI 0.162 el servidor de Prefect no arrancaba; se fijó `fastapi<0.142` ([D2](reports/decisiones-e-ia.md#d2-fastapi0142-por-incompatibilidad-con-prefect-388)).


# $\color{#dda}{\text{7. API}}$

[api/main.py](api/main.py). Al iniciar resuelve `champion` → número de versión y carga **esa versión concreta**; la conserva mientras corre, y un cambio de alias se adopta al reiniciar ([D8](reports/decisiones-e-ia.md#d8-la-api-fija-la-versión-al-iniciar-y-valida-con-pydantic-estricto)).

| Operación | Comportamiento |
|---|---|
| `GET /health` | 200 con `status`, `model_name`, `model_alias`, `model_version` sólo si el modelo está cargado; 503 si no |
| `POST /predict` | `{"text": "..."}` → 200 con `label`, `model_name`, `model_version` (cadenas) |
| Validación | `extra="forbid"`, `strict=True`, 1–2,000 caracteres, no sólo espacios → 422 |

Casos inválidos verificados (evidencias `07-*`): `{}`, `{"text":7}`, sólo espacios, 2,001 caracteres y `{"text":"Hello","label":"ham"}` responden **422**; 2,000 caracteres exactos responde 200.

**Paridad** ([src/paridad.py](src/paridad.py)): tres textos inventados, predicción directa con el modelo cargado del registro contra la respuesta de la API.


In [17]:
import json
paridad = json.loads(Path("reports/paridad.json").read_text(encoding="utf-8"))
print("Versión:", paridad["version"], "| todas coinciden:", paridad["todas_coinciden"])
pd.DataFrame(paridad["casos"])

Versión: 1 | todas coinciden: True


,texto,modelo,api,http,version_api,coinciden
0,Can we meet after class to review the project notes?,ham,ham,200,1,True
1,"URGENT! You have won a 500 cash prize. Text WIN to 80888 now to claim, T&Cs ...",spam,spam,200,1,True
2,"Your package is waiting, reply to confirm the delivery time for tomorrow.",spam,spam,200,1,True


# $\color{#dda}{\text{8. Interfaz y contenedores}}$

**Recorrido del usuario** ([ui/app.py](ui/app.py)): pega el SMS → **Clasificar** → ve la estimación (`ham`/`spam`) y la versión del modelo. La interfaz avisa que trabaja con inglés y que es una estimación. Si el texto es inválido muestra el motivo; si la API no responde, un mensaje claro en vez de un error técnico. No carga el modelo ni tiene credenciales: sólo conoce `API_URL`.

**Contenedores** ([api/Dockerfile](api/Dockerfile), [ui/Dockerfile](ui/Dockerfile), [compose.yaml](compose.yaml)):

- Contexto de build = raíz del repo, porque ambas imágenes copian `pyproject.toml`, `uv.lock` y `.python-version`; cada una instala **sólo su grupo** con `uv sync --locked --only-group` equivalente (`--no-default-groups --group api|ui`). La UI no contiene MLflow ni scikit-learn.
- Usuario sin privilegios y sin caché de uv: 0.95 GB (API) y 0.88 GB (UI); la primera versión pesaba 2 GB por un `chown -R` que duplicaba el ambiente.
- `.dockerignore` excluye `.env`, `.git/`, `data/`, `artifacts/`. El `.env` llega **sólo** al contenedor de la API en tiempo de ejecución (`env_file`).
- Red *bridge* de Compose: la UI llama a `http://api:8060`; los puertos se publican en `127.0.0.1` ([D9](reports/decisiones-e-ia.md#d9-contenedores-red-bridge-en-vez-de-host)). La UI espera a que la API esté `healthy`.

```bash
docker compose build
docker compose up -d
docker compose ps            # sms-api-container (healthy), sms-ui-container
# http://127.0.0.1:8560
docker compose stop api      # la UI muestra "No se pudo conectar con la API"
docker compose down          # cierre comprobado con docker compose ps
```

Para la sustentación, el mismo Compose corre en mi computadora de escritorio (Linux en WSL) y se accede por mi red privada de Tailscale, sin abrir puertos a Internet. Se apaga al terminar el curso.


# $\color{#dda}{\text{9. AWS}}$

Explicación completa en [reports/despliegue-aws.md](reports/despliegue-aws.md); no se desplegó (opcional). Resumen de decisiones:

- EC2 del laboratorio en `us-east-1`, Ubuntu LTS, `t3.small` y 20 GiB gp3 (construir imágenes con 1 GiB de RAM es arriesgado).
- SSH sólo desde **My IP**; clon del repo privado con una *deploy key* de sólo lectura; `.env` creado en el servidor con `chmod 600`.
- Mismo `compose.yaml`; la UI se publica en el puerto 8560 sólo para mi IP, la API no se expone.
- Verificación con `curl` a `/health` y `/predict` desde la sesión SSH y la UI desde mi navegador.
- Costos: ~0.02 USD/h de cómputo; el disco, la IP elástica y los snapshots siguen cobrando con la instancia detenida. **Detener** conserva el disco; **terminar** lo elimina.


# $\color{#dda}{\text{10. Decisiones e IA}}$

La bitácora completa está en [reports/decisiones-e-ia.md](reports/decisiones-e-ia.md). Decisiones clave: deduplicar antes de partir (D3), TF-IDF dentro del modelo registrado (D4), F1 de spam con desempate por precision definida antes del tuning (D5), no cambiar el champion por resultados de prueba (D6) ni retunear tras verlos (D7).

Usé Claude Code como asistente para escribir código y documentación. Cada uso tiene su prompt original, qué acepté o rechacé y cómo lo comprobé con ejecuciones propias (conteos contra el enunciado, flujo completo, casos HTTP, paridad). No compartí mensajes del corpus ni credenciales con la IA: el análisis de errores lo hice leyendo los mensajes localmente.


# $\color{#dda}{\text{11. Conclusiones y referencias}}$

**Respuesta al problema.** El prototipo clasifica SMS en inglés con F1 de spam de 0.947 en prueba (precision 0.983, recall 0.914) usando un modelo pequeño y explicable, servido con la versión exacta que se evaluó y consultable desde una interfaz. Todo el recorrido se reproduce con dos comandos (`uv run python src/train_flow.py` y `docker compose up`).

**Errores.** El modelo es conservador: 2 FP contra 11 FN. Para una herramienta que sólo sugiere, es preferible a lo contrario; si el uso fuera filtrar automáticamente, convendría mover el umbral o elegir por recall.

**Desbalance.** Con 12 % de spam, accuracy no sirve para comparar (el baseline tiene 0.876). Reponderar las clases no mejoró F1 en validación; la información útil está en los términos.

**Límites del corpus.** Mensajes de 2011, en inglés y en su mayoría del Reino Unido y Singapur: no cubre fraudes actuales (paquetería, bancos falsos, enlaces acortados), otros idiomas ni mensajes con emojis. Un texto inventado de paquetería fue clasificado como spam, lo cual es razonable, pero el corpus no permite medir qué tan bien detecta esas campañas.

**Mejoras.** Validación cruzada para estimar la varianza entre finalistas; ampliar el rango de `C`; probar embeddings multilingües; reunir mensajes recientes etiquetados para evaluar el desgaste del modelo.

**Referencias.** Almeida, T. y Hidalgo, J. (2011). *SMS Spam Collection* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5CC84 (CC BY 4.0). Documentación de scikit-learn, Optuna, MLflow, Prefect, FastAPI y Streamlit. Material del curso PCD (ITESO, otoño 2026) y mis tareas 5 y 6.

### Índice de evidencias

| Caso | Archivo | Resultado observado |
|---|---|---|
| PR e historial | `reports/evidencias/01-pr-merged.png` | PR abierto hacia `main` con sus commits; el estado *Merged* se verifica en la URL entregada en Canvas |
| Ambiente reconstruido | `reports/evidencias/02-uv-sync-locked.png` | `uv sync --locked` sin cambios |
| Depuración y particiones | `reports/evidencias/03-datos-conteos.png` | 5,574 → 5,159; 0 claves compartidas |
| Gráficas de clases y longitud | Secciones 2 de este informe | Interpretadas arriba |
| Comparación y matriz | `reports/evidencias/04-evaluacion.png` | Baseline/finalistas; matriz 902/2/11/117 |
| Comparación de experimentos | `reports/evidencias/05-mlflow-comparacion.png` | 40 trials en dos estudios |
| Detalle de un trial | `reports/evidencias/05-mlflow-trial.png` | Parámetros y métricas `val_*` |
| Registro con aliases | `reports/evidencias/05-registro-versiones.png` | v1 champion, v2 challenger con origen |
| Flujo Prefect | `reports/evidencias/06-prefect-flujo.png` | Tasks completadas, estado final `Completed` |
| Salida del flujo en terminal | `reports/evidencias/06-prefect-terminal.png` | Resumen champion/challenger |
| Salud de la API | `reports/evidencias/07-api-health.png` | 200 con versión 1 |
| Predicción válida | `reports/evidencias/07-api-predict.png` | 200 con `label` y versión |
| Paridad | `reports/evidencias/07-api-paridad.png` | 3/3 coinciden |
| 422: falta `text` | `reports/evidencias/07-api-422-vacio.png` | 422 `missing` |
| 422: tipo incorrecto | `reports/evidencias/07-api-422-tipo.png` | 422 `string_type` |
| 422: sólo espacios | `reports/evidencias/07-api-422-espacios.png` | 422 `value_error` |
| 422: 2,001 caracteres | `reports/evidencias/07-api-422-largo.png` | 422 `string_too_long` |
| 422: campo adicional | `reports/evidencias/07-api-422-extra.png` | 422 `extra_forbidden` |
| UI con predicción | `reports/evidencias/08-ui-prediccion.png` | Etiqueta y versión |
| UI con API detenida | `reports/evidencias/08-ui-api-detenida.png` | Mensaje de conexión |
| Build de imágenes | `reports/evidencias/09-docker-build.png` | Ambas imágenes construidas |
| Servicios activos y predicción | `reports/evidencias/09-docker-ps.png` | API healthy, UI arriba |
| Cierre | `reports/evidencias/09-docker-down.png` | Sin contenedores tras `down` |
